# Distance-Based Neighbourhood Graph Feature Analysis

## Imports and General Setup

In [ ]:
import csv
import pickle
import numpy as np
from skimage import io
from scipy.ndimage import binary_fill_holes

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
TARGET_SCALE = "min_234"
MIN_CELL_AREA = 10000

DATASET_CONFIGS = {
    1: {
        "image_path": "Lysosome_masked_crops_1024.tif",
        "csv_path": "PhTau_histogram_clusters.csv",
        "coord_path": "processed_data/spot_coordinates.pkl",
        "delaunay_path": "processed_data/delaunay_results.pkl",
    },
    2: {
        "image_path": "cell_gallery_1024x1024_6ch.tif",
        "csv_path": "gallery_phtau_ground_truth.csv",
        "coord_path": "processed_data_2/spot_coordinates.pkl",
        "delaunay_path": "processed_data_2/delaunay_results.pkl",
    },
}

# Master data containers
images = {}
n_images = {}
labels = {}
categories = {}
coord_map = {}
delaunay_objects_dict = {}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_csv_labels(csv_path, total_expected_frames):
    """Parses cluster labels from CSV and pads remaining frames as untreated."""
    parsed_labels = []
    with open(csv_path, "r") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header row
        for row in reader:
            val = row[-1]
            if val == "0":
                parsed_labels.append(["0", "dox+\nPhTau+"])
            elif val == "1":
                parsed_labels.append(["1", "dox+\nPhTau-"])

    # Pad missing indices up to the total stack size (assumed untreated)
    while len(parsed_labels) < total_expected_frames:
        parsed_labels.append(["2", "dox-"])
        
    return parsed_labels


def load_pickle_scale(pickle_path, scale_key):
    """Helper to extract a specific resolution scale from a pickle file."""
    with open(pickle_path, "rb") as f:
        return pickle.load(f)[scale_key]


def compute_valid_mask_indices(image_stack, area_threshold):
    """ Vectorised masking and area filter over a standardised image stack."""
    # Since channel is guaranteed to be at axis -1, collapse it if 4D
    raw_masks = np.any(image_stack > 0, axis=-1) if image_stack.ndim == 4 else image_stack > 0

    # Fill internal segmentation holes slice-by-slice across the stack
    cell_masks = binary_fill_holes(raw_masks)
    
    # Calculate pixel surface areas by summing spatial dimensions (axes 1 and 2)
    cell_areas = np.sum(cell_masks, axis=(1, 2))
    
    # Return boolean array masking frames that meet size criteria
    return cell_areas >= area_threshold

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================
for idx, config in DATASET_CONFIGS.items():
    # Load image stack pipeline
    raw_images_stack = io.imread(config["image_path"])

    # Standardise 4D stacks to always be (frames, spatial_1, spatial_2, channels)
    if raw_images_stack.ndim == 4:
        # If your channel is currently at axis 1 (e.g., shape 100, 6, 1024, 1024)
        channel_axis = np.argmin(raw_images_stack.shape)
        if channel_axis != -1:
            raw_images_stack = np.moveaxis(raw_images_stack, channel_axis, -1)
    
    total_frames = raw_images_stack.shape[0]

    # Load and map structural metadata sequentially
    all_labels = load_csv_labels(config["csv_path"], total_frames)
    full_coord = load_pickle_scale(config["coord_path"], TARGET_SCALE)
    full_delaunay = load_pickle_scale(config["delaunay_path"], TARGET_SCALE)

    # Compute valid mask filter indices using vectorised area sums
    valid_indices = compute_valid_mask_indices(raw_images_stack, MIN_CELL_AREA)

    # Identify the rejected indices where the condition is False
    rejected_indices = np.where(~valid_indices)[0]
    
    # Print the rejected cell indices for the current dataset
    print(f"Dataset {idx} ({config['image_path']}):")
    if len(rejected_indices) > 0:
        print(f"  Rejected {len(rejected_indices)} cells due to small area (< {MIN_CELL_AREA} px).")
        print(f"  Rejected Indices: {rejected_indices.tolist()}\n")
    else:
        print("  No cells were rejected.\n")

    # Apply boolean index filtering to numerical arrays
    images[idx] = raw_images_stack[valid_indices]
    n_images[idx] = np.sum(valid_indices)
    
    # Filter Python list metadata using matching list comprehensions
    labels[idx] = [all_labels[i] for i, valid in enumerate(valid_indices) if valid]
    coord_map[idx] = [full_coord[i] for i, valid in enumerate(valid_indices) if valid]
    delaunay_objects_dict[idx] = [full_delaunay[i] for i, valid in enumerate(valid_indices) if valid]

    # Parse remaining categories, sorting by category id descending
    unique_pairs = {tuple(item) for item in labels[idx]}
    sorted_pairs = sorted(unique_pairs, key=lambda x: int(x[0]), reverse=True)
    categories[idx] = [pair[1] for pair in sorted_pairs]

In [ ]:
from itertools import combinations

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests


def run_group_tests(
    df,
    value_col,
    dataset_name,
    metric_name,
    condition_order,
    group_col="Condition",
    alpha=0.05
):
    """
    Run a Kruskal–Wallis test followed by pairwise Mann–Whitney U tests.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame.
    value_col : str
        Numeric measurement column.
    dataset_name : str
        Dataset name printed in the output.
    metric_name : str
        Human-readable measurement name.
    condition_order : list
        Desired order of groups.
    group_col : str, default="Condition"
        Column containing the group labels.
    alpha : float, default=0.05
        Significance threshold.
    """

    # Validate required columns
    missing_columns = [
        column for column in [value_col, group_col]
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing column(s): {missing_columns}. "
            f"Available columns are: {list(df.columns)}"
        )

    print(f"\n{'=' * 75}")
    print(f"{metric_name} — {dataset_name}")
    print(f"{'=' * 75}")

    # Extract non-empty groups in the requested order
    groups = {
        group: df.loc[
            df[group_col] == group,
            value_col
        ].dropna()
        for group in condition_order
    }

    # Remove groups with no observations
    groups = {
        group: values
        for group, values in groups.items()
        if len(values) > 0
    }

    print("\nSample sizes:")
    for group, values in groups.items():
        print(f"  {group}: n = {len(values)}")

    if len(groups) < 2:
        print("\nNot enough groups for statistical testing.")
        return None

    # Kruskal–Wallis omnibus test
    kw_stat, kw_p = kruskal(*groups.values())

    print("\nKruskal–Wallis test:")
    print(f"  H = {kw_stat:.4f}")
    print(f"  p = {kw_p:.4g}")

    if kw_p < alpha:
        print(f"  Significant overall difference at alpha = {alpha}.")
    else:
        print(f"  No significant overall difference at alpha = {alpha}.")

    # Pairwise Mann–Whitney U tests
    pairwise_results = []

    for group_a, group_b in combinations(groups.keys(), 2):
        values_a = groups[group_a]
        values_b = groups[group_b]

        u_stat, raw_p = mannwhitneyu(
            values_a,
            values_b,
            alternative="two-sided",
            method="asymptotic"
        )

        pairwise_results.append({
            "Metric": metric_name,
            "Dataset": dataset_name,
            "Group column": group_col,
            "Comparison": f"{group_a} vs {group_b}",
            "U": u_stat,
            "Raw_p": raw_p
        })

    # Holm correction for multiple comparisons
    raw_p_values = [
        result["Raw_p"]
        for result in pairwise_results
    ]

    _, adjusted_p_values, _, _ = multipletests(
        raw_p_values,
        method="holm",
        alpha=alpha
    )

    for result, adjusted_p in zip(
        pairwise_results,
        adjusted_p_values
    ):
        result["Holm_adjusted_p"] = adjusted_p
        result["Significant"] = adjusted_p < alpha

    results_df = pd.DataFrame(pairwise_results)

    print("\nPairwise Mann–Whitney U tests with Holm correction:")
    print(
        results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    return results_df

## Build Graphs

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx

from scipy.spatial import KDTree
from scipy.ndimage import binary_fill_holes


# ============================================================
# CONFIGURATION
# ============================================================

PIXEL_WIDTH = 0.0739859  # micrometres per pixel
PIXEL_AREA = PIXEL_WIDTH ** 2  # square micrometres per pixel^2

RADIUS_MULTIPLIERS = [1.0, 1.5, 2.0, 2.5]


# ============================================================
# POINT AND DISTANCE FUNCTIONS
# ============================================================

def clean_points(points):
    """
    Return finite coordinates as an N x 2 float array in micrometres.

    Parameters
    ----------
    points : array-like
        Coordinates in pixel units with shape (N, 2).

    Returns
    -------
    numpy.ndarray
        Cleaned coordinates in micrometres with shape (N, 2).
    """
    points = np.asarray(points, dtype=float)

    if points.size == 0:
        return np.empty((0, 2), dtype=float)

    points = np.atleast_2d(points)

    if points.shape[1] != 2:
        raise ValueError(
            f"Expected coordinates with shape (N, 2), "
            f"but received {points.shape}."
        )

    finite_rows = np.all(np.isfinite(points), axis=1)

    # Convert pixel coordinates to micrometres.
    points_um = points[finite_rows] * PIXEL_WIDTH

    return points_um


def nearest_neighbour_distances(points):
    """
    Calculate raw nearest-neighbour distances for one cell.

    Input coordinates are assumed to be in pixels.
    Returned distances are in micrometres.
    """
    points_um = clean_points(points)

    if len(points_um) < 2:
        return np.array([], dtype=float)

    tree = KDTree(points_um)

    distances, _ = tree.query(points_um, k=2)

    # The first distance is each point's distance to itself.
    return distances[:, 1]


def expected_random_nn_distance(n_points, cell_area_um2):
    """
    Expected nearest-neighbour distance for a homogeneous random
    two-dimensional point pattern.

    Parameters
    ----------
    n_points : int
        Number of lysosomes.
    cell_area_um2 : float
        Cell area in square micrometres.

    Returns
    -------
    float
        Expected nearest-neighbour distance in micrometres.
    """
    if (
        n_points <= 0
        or not np.isfinite(cell_area_um2)
        or cell_area_um2 <= 0
    ):
        return np.nan

    density_per_um2 = n_points / cell_area_um2

    if not np.isfinite(density_per_um2) or density_per_um2 <= 0:
        return np.nan

    return 1.0 / (2.0 * np.sqrt(density_per_um2))


def normalised_nearest_neighbour_distances(
    points,
    cell_area_um2
):
    """
    Density-normalise nearest-neighbour distances.

    Input coordinates are assumed to be in pixels.
    Cell area must be supplied in square micrometres.

    Values below 1 indicate shorter-than-expected distances.
    Values above 1 indicate longer-than-expected distances.
    """
    points_um = clean_points(points)

    raw_nn_distances_um = nearest_neighbour_distances(
        points_um
    )

    if len(raw_nn_distances_um) == 0:
        return np.array([], dtype=float)

    expected_distance_um = expected_random_nn_distance(
        n_points=len(points_um),
        cell_area_um2=cell_area_um2
    )

    if (
        not np.isfinite(expected_distance_um)
        or expected_distance_um <= 0
    ):
        return np.full(
            len(raw_nn_distances_um),
            np.nan
        )

    return raw_nn_distances_um / expected_distance_um


# ============================================================
# GRAPH CREATION FUNCTION
# ============================================================

def build_radius_graph(points, radius_um):
    """
    Construct an undirected radius-neighbourhood graph.

    Parameters
    ----------
    points : array-like
        Coordinates in pixel units.
    radius_um : float
        Connection radius in micrometres.

    Returns
    -------
    networkx.Graph
        Radius-neighbourhood graph.
    """
    points_um = clean_points(points)

    n_points = len(points_um)

    graph = nx.Graph()

    # Add one node per cleaned lysosome coordinate.
    graph.add_nodes_from(range(n_points))

    if n_points < 2:
        return graph

    if not np.isfinite(radius_um) or radius_um <= 0:
        return graph

    tree = KDTree(points_um)

    neighbours = tree.query_ball_point(
        points_um,
        r=radius_um
    )

    for i, neighbour_indices in enumerate(neighbours):

        for j in neighbour_indices:

            # Prevent self-edges and duplicate undirected edges.
            if i < j:
                graph.add_edge(i, j)

    return graph


# ============================================================
# LABEL AND AREA FUNCTIONS
# ============================================================

def calculate_cell_area(image):
    """
    Calculate the filled two-dimensional cell-mask area.

    The input image or mask is assumed to be represented in pixels.
    The returned area is in square micrometres.

    Supports:
        - a 2D mask/image
        - a multichannel image, where any positive channel
          indicates cell pixels
    """
    image = np.asarray(image)

    if image.ndim == 3:
        mask = np.any(image > 0, axis=-1)

    elif image.ndim == 2:
        mask = image > 0

    else:
        raise ValueError(
            f"Unexpected image shape: {image.shape}"
        )

    filled_mask = binary_fill_holes(mask)

    area_pixels = np.sum(filled_mask)

    area_um2 = area_pixels * PIXEL_AREA

    return float(area_um2)


# ============================================================
# PRECOMPUTE CELL-LEVEL DATA
# ============================================================

cell_data = {}

for dataset_id in DATASET_CONFIGS.keys():

    cell_data[dataset_id] = {}

    for image_id in range(n_images[dataset_id]):

        tau_state = labels[dataset_id][image_id][1]

        # Input coordinates are in pixels.
        points_pixels = coord_map[dataset_id][image_id]

        # Coordinates are cleaned and converted to micrometres.
        points_um = clean_points(points_pixels)

        cell_area_um2 = calculate_cell_area(
            images[dataset_id][image_id]
        )

        n_lysosomes = len(points_um)

        lysosome_density_per_um2 = (
            n_lysosomes / cell_area_um2
            if cell_area_um2 > 0
            else np.nan
        )

        raw_nn_distances_um = nearest_neighbour_distances(
            points_pixels
        )

        normalised_nn_distances = (
            normalised_nearest_neighbour_distances(
                points=points_pixels,
                cell_area_um2=cell_area_um2
            )
        )

        cell_data[dataset_id][image_id] = {
            "tau_state": tau_state,
            "label": labels[dataset_id][image_id][1],

            # Coordinates are stored in micrometres.
            "points": points_um,
            "points_um": points_um,

            # Area and density use physical units.
            "cell_area": cell_area_um2,
            "cell_area_um2": cell_area_um2,
            "n_lysosomes": n_lysosomes,
            "lysosome_density": lysosome_density_per_um2,
            "lysosome_density_per_um2": (
                lysosome_density_per_um2
            ),

            # Distances are in micrometres.
            "raw_nn_distances": raw_nn_distances_um,
            "raw_nn_distances_um": raw_nn_distances_um,
            "normalised_nn_distances": (
                normalised_nn_distances
            ),
        }


# ============================================================
# CALCULATE DATASET-LEVEL COMMON RADII
# ============================================================

def calculate_dataset_radii(points_list, multipliers):
    """
    Calculate one common set of raw spatial radii per dataset.

    Parameters
    ----------
    points_list : list
        List of coordinate arrays. Coordinates may be supplied
        in pixels or micrometres, provided that the same units
        are used consistently.

        In this workflow, the arrays stored in cell_data are
        already converted to micrometres.

    multipliers : iterable of float
        Multipliers applied to the pooled median nearest-neighbour
        distance.

    Returns
    -------
    dict
        Mapping from multiplier to radius in micrometres.
    """
    all_nn_distances_um = []

    for points_um in points_list:

        # points_um are already in micrometres.
        nn_distances_um = nearest_neighbour_distances(
            points_um / PIXEL_WIDTH
        )

        all_nn_distances_um.extend(
            nn_distances_um
        )

    all_nn_distances_um = np.asarray(
        all_nn_distances_um,
        dtype=float
    )

    all_nn_distances_um = all_nn_distances_um[
        np.isfinite(all_nn_distances_um)
    ]

    if len(all_nn_distances_um) == 0:
        return {
            multiplier: np.nan
            for multiplier in multipliers
        }

    reference_distance_um = np.median(
        all_nn_distances_um
    )

    return {
        multiplier: reference_distance_um * multiplier
        for multiplier in multipliers
    }


common_radii = {}

for dataset_id in DATASET_CONFIGS.keys():

    included_cells = cell_data[dataset_id].values()

    dataset_points_um = [
        cell["points_um"]
        for cell in included_cells
    ]

    radii_um = calculate_dataset_radii(
        points_list=dataset_points_um,
        multipliers=RADIUS_MULTIPLIERS
    )

    common_radii[dataset_id] = radii_um

    print(f"\nDataset {dataset_id}")

    for multiplier, radius_um in radii_um.items():

        print(
            f"  multiplier = {multiplier:.2f}, "
            f"radius = {radius_um:.3f} µm"
        )


# ============================================================
# CREATE AND STORE GRAPHS ONLY
# ============================================================

graphs = {}

for dataset_id in DATASET_CONFIGS.keys():

    graphs[dataset_id] = {}

    radii_um = common_radii[dataset_id]

    for image_id, cell in cell_data[dataset_id].items():

        points_um = cell["points_um"]

        graphs[dataset_id][image_id] = {}

        for multiplier, radius_um in radii_um.items():

            # build_radius_graph expects pixel coordinates and
            # converts them to micrometres internally.
            points_pixels = points_um / PIXEL_WIDTH

            graph = build_radius_graph(
                points=points_pixels,
                radius_um=radius_um
            )

            # Store the graph only.
            # No graph metrics are calculated in this cell.
            graphs[dataset_id][image_id][multiplier] = graph


print("\nGraph creation complete.")

n_stored_graphs = sum(
    len(cell_graphs)
    for dataset_graphs in graphs.values()
    for cell_graphs in dataset_graphs.values()
)

print(f"Number of stored graphs: {n_stored_graphs}")

In [ ]:
import os
import matplotlib.pyplot as plt
import networkx as nx

# Select the 2× radius graph of the first cell in Dataset 1
dataset_id = 1
image_id = 0
radius_multiplier = 2.0

first_graph = graphs[dataset_id][image_id][radius_multiplier]

# Retrieve the corresponding lysosome coordinates in micrometres
points_um = cell_data[dataset_id][image_id]["points_um"]

# Assign positions to graph nodes
positions = {
    node_id: points_um[node_id]
    for node_id in first_graph.nodes
}

# Obtain the physical radius used for this graph
radius_um = common_radii[dataset_id][radius_multiplier]

# Plot the radius graph
fig, ax = plt.subplots(figsize=(4, 4))

nx.draw(
    first_graph,
    pos=positions,
    ax=ax,
    node_size=10,
    node_color="red",
    edge_color="black",
    width=0.6,
    alpha=0.8,
)

ax.set_title(
    f"Distance-based graph ({radius_multiplier:g}× NN distance)"
)

ax.set_aspect("equal")
ax.invert_yaxis()  # Match image-coordinate convention

# Remove axis numbers and ticks
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()

os.makedirs("Figures", exist_ok=True)

plt.savefig(
    "Figures/radius_graph_2x.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()

## Analysis

In [ ]:
# ============================================================
# GRAPH METRIC FUNCTION
# ============================================================

def graph_metrics(graph):
    """Calculate graph-level metrics for one cell."""

    n_nodes = graph.number_of_nodes()

    if n_nodes == 0:
        return {
            "n_edges": 0,
            "mean_degree": np.nan,
            "graph_density": np.nan,
            "isolated_fraction": np.nan,
            "n_components": 0,
            "largest_component_fraction": np.nan,
            "transitivity": np.nan,
        }

    degrees = np.array(
        [degree for _, degree in graph.degree()],
        dtype=float
    )

    component_sizes = [
        len(component)
        for component in nx.connected_components(graph)
    ]

    return {
        "n_edges": graph.number_of_edges(),
        "mean_degree": np.mean(degrees),
        "graph_density": nx.density(graph),
        "isolated_fraction": np.mean(degrees == 0),
        "n_components": len(component_sizes),
        "largest_component_fraction": (
            max(component_sizes) / n_nodes
            if component_sizes
            else np.nan
        ),
        "transitivity": (
            nx.transitivity(graph)
            if n_nodes >= 3
            else np.nan
        ),
    }


# ============================================================
# CALCULATE CELL-LEVEL GRAPH METRICS
# ============================================================

graph_results_list = []

for dataset_id, dataset_graphs in graphs.items():

    for image_id, cell_graphs in dataset_graphs.items():

        cell = cell_data[dataset_id][image_id]

        for multiplier, graph in cell_graphs.items():

            # Calculate graph-level metrics for this cell and radius.
            metrics = graph_metrics(graph)

            result = {
                "dataset": dataset_id,
                "cell_id": image_id,
                "tau_state": cell["tau_state"],
                "label": cell["label"],
                "radius_multiplier": multiplier,

                # Radius is expressed in micrometres.
                "radius_um": common_radii[
                    dataset_id
                ][multiplier],
            }

            result.update(metrics)
            graph_results_list.append(result)


graph_results = pd.DataFrame(graph_results_list)


# ============================================================
# BASIC CHECKS
# ============================================================

print("Graph results:")
print(graph_results.head())

print("\nGraph-results shape:")
print(graph_results.shape)

print("\nNumber of unique cells by dataset and Tau state:")
print(
    graph_results
    .drop_duplicates(["dataset", "cell_id"])
    .groupby(["dataset", "tau_state"])
    .size()
)

print("\nNumber of graph observations by radius:")
print(
    graph_results
    .groupby(
        [
            "dataset",
            "tau_state",
            "radius_multiplier",
            "radius_um",
        ]
    )
    .size()
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

os.makedirs("Figures", exist_ok=True)

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    """Convert condition labels to a consistent format."""
    if pd.isna(label):
        return label

    return label_map.get(
        str(label),
        str(label)
    )


# ==========================================
# 2. PREPARE DATAFRAMES
# ==========================================

plot_results = graph_results.copy()

# Use the original label where available.
# Fall back to tau_state if label is missing.
plot_results["Condition"] = (
    plot_results["label"]
    .fillna(plot_results["tau_state"])
    .apply(standardise_condition)
)

# Keep only datasets used for the original geometric-feature plots.
# Dataset labels may be integers or strings.
dataset_1_values = [1, "1"]
dataset_2_values = [2, "2"]

df1 = plot_results[
    plot_results["dataset"].isin(dataset_1_values)
].copy()

df2 = plot_results[
    plot_results["dataset"].isin(dataset_2_values)
].copy()

# Construct the combined dataset without modifying graph_results.
df_combined = pd.concat(
    [df1, df2],
    ignore_index=True
)

print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)
print("Combined dataset shape:", df_combined.shape)

print("\nConditions in Dataset 1:")
print(
    df1["Condition"].value_counts(
        dropna=False
    )
)

print("\nConditions in Dataset 2:")
print(
    df2["Condition"].value_counts(
        dropna=False
    )
)


# ==========================================
# 3. GRAPH METRICS TO PLOT
# ==========================================

plot_definitions = [
    (
        "n_edges",
        "Number of Graph Edges",
        "Number of edges",
        "number_of_edges"
    ),
    (
        "mean_degree",
        "Mean Degree",
        "Mean degree",
        "mean_degree"
    ),
    (
        "graph_density",
        "Graph Density",
        "Graph density",
        "graph_density"
    ),
    (
        "isolated_fraction",
        "Fraction of Isolated Lysosomes",
        "Isolated lysosome fraction",
        "isolated_fraction"
    ),
    (
        "n_components",
        "Number of Connected Components",
        "Number of connected components",
        "number_of_components"
    ),
    (
        "largest_component_fraction",
        "Largest Component Fraction",
        "Largest component fraction",
        "largest_component_fraction"
    ),
    (
        "transitivity",
        "Graph Transitivity",
        "Transitivity",
        "transitivity"
    )
]


# ==========================================
# 4. HELPER FUNCTIONS
# ==========================================

def radius_filename(radius_multiplier):
    """
    Convert a radius multiplier into a filename-safe string.

    For example, 1.5 becomes radius_1p5.
    """
    value = str(radius_multiplier)
    value = value.replace(".", "p")
    value = re.sub(
        r"[^A-Za-z0-9_\-]",
        "_",
        value
    )

    return f"radius_{value}"


def radius_title(df, radius_multiplier):
    """
    Create a readable title for a radius multiplier.

    The radius is reported in micrometres.
    """
    radius_subset = df[
        df["radius_multiplier"] == radius_multiplier
    ]

    if (
        "radius_um" in radius_subset.columns
        and radius_subset["radius_um"].notna().any()
    ):
        radius_value_um = (
            radius_subset["radius_um"]
            .dropna()
            .iloc[0]
        )

        return (
            f"radius multiplier = {radius_multiplier}, "
            f"radius = {radius_value_um:.2f} µm"
        )

    return (
        f"radius multiplier = "
        f"{radius_multiplier}"
    )


def available_order(df, preferred_order):
    """Return only condition labels present in the dataframe."""
    present_conditions = (
        df["Condition"]
        .dropna()
        .unique()
    )

    return [
        condition
        for condition in preferred_order
        if condition in present_conditions
    ]


def plot_violin(
    ax,
    df,
    column,
    order
):
    """Draw one condition-grouped violin plot."""

    df_plot = df.dropna(
        subset=[
            column,
            "Condition"
        ]
    ).copy()

    if df_plot.empty or len(order) == 0:
        ax.text(
            0.5,
            0.5,
            "No data available",
            ha="center",
            va="center",
            transform=ax.transAxes
        )

        ax.set_xticks([])

        return

    sns.violinplot(
        ax=ax,
        data=df_plot,
        x="Condition",
        y=column,
        hue="Condition",
        order=order,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_xlabel(
        "Condition",
        fontsize=12
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.5
    )

    ax.tick_params(
        axis="x",
        rotation=0
    )


# ==========================================
# 5. CREATE PLOTS FOR ONE METRIC AND RADIUS
# ==========================================

def create_violin_plots(
    column,
    title,
    ylabel,
    filename_prefix,
    radius_multiplier
):
    """
    Create and save:

    1. Dataset 1 versus Dataset 2 plot
    2. Combined dataset plot

    Each plot is generated for one radius multiplier.
    """

    df1_radius = df1[
        df1["radius_multiplier"] == radius_multiplier
    ].copy()

    df2_radius = df2[
        df2["radius_multiplier"] == radius_multiplier
    ].copy()

    df_combined_radius = df_combined[
        df_combined["radius_multiplier"] == radius_multiplier
    ].copy()

    order_1 = available_order(
        df1_radius,
        full_order
    )

    order_2 = available_order(
        df2_radius,
        full_order
    )

    radius_label = radius_title(
        graph_results,
        radius_multiplier
    )

    # --------------------------------------
    # Plot 1: Dataset 1 and Dataset 2
    # --------------------------------------

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 3),
        sharey=True
    )

    plot_violin(
        ax=axes[0],
        df=df1_radius,
        column=column,
        order=order_1
    )

    plot_violin(
        ax=axes[1],
        df=df2_radius,
        column=column,
        order=order_2
    )

    axes[0].set_title(
        f"{title} (Dataset 1)\n{radius_label}",
        fontsize=13,
        pad=10
    )

    axes[1].set_title(
        f"{title} (Dataset 2)\n{radius_label}",
        fontsize=13,
        pad=10
    )

    axes[0].set_ylabel(
        ylabel,
        fontsize=12
    )

    axes[1].set_ylabel("")

    fig.tight_layout()

    filename = (
        f"Figures/"
        f"{filename_prefix}_"
        f"{radius_filename(radius_multiplier)}_"
        f"by_dataset.svg"
    )

    fig.savefig(
        filename,
        format="svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # --------------------------------------
    # Plot 2: Combined Dataset
    # --------------------------------------

    order_combined = available_order(
        df_combined_radius,
        full_order
    )

    fig, ax = plt.subplots(
        figsize=(5, 3)
    )

    plot_violin(
        ax=ax,
        df=df_combined_radius,
        column=column,
        order=order_combined
    )

    ax.set_title(
        f"{title}\n{radius_label}",
        fontsize=13,
        pad=10
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12
    )

    fig.tight_layout()

    filename = (
        f"Figures/"
        f"{filename_prefix}_"
        f"{radius_filename(radius_multiplier)}_"
        f"combined.svg"
    )

    fig.savefig(
        filename,
        format="svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ==========================================
# 6. CREATE ALL FIGURES
# ==========================================

radius_multipliers = sorted(
    graph_results[
        "radius_multiplier"
    ]
    .dropna()
    .unique()
)

for column, title, ylabel, filename_prefix in (
    plot_definitions
):

    if column not in graph_results.columns:
        print(
            f"Skipping '{column}': "
            "column not found in graph_results."
        )

        continue

    for multiplier in radius_multipliers:

        create_violin_plots(
            column=column,
            title=title,
            ylabel=ylabel,
            filename_prefix=filename_prefix,
            radius_multiplier=multiplier
        )


# ==========================================
# 7. SUMMARY STATISTICS
# ==========================================

summary_columns = [
    column
    for column, _, _, _ in plot_definitions
    if column in graph_results.columns
]


def print_summary(
    df,
    dataset_name
):
    print(
        f"\n--- {dataset_name} "
        "Summary Statistics ---"
    )

    for multiplier in radius_multipliers:

        df_radius = df[
            df["radius_multiplier"] == multiplier
        ].copy()

        print(
            f"\nRadius multiplier: "
            f"{multiplier}"
        )

        summary = (
            df_radius
            .groupby("Condition")[summary_columns]
            .describe()
            .round(3)
        )

        print(summary)


print_summary(
    df1,
    "Dataset 1"
)

print_summary(
    df2,
    "Dataset 2"
)

print_summary(
    df_combined,
    "Combined Dataset"
)


# ==========================================
# 8. OPTIONAL: FLAT MEDIAN TABLE
# ==========================================

median_summary = (
    graph_results
    .assign(
        Condition=(
            graph_results["label"]
            .fillna(graph_results["tau_state"])
            .apply(standardise_condition)
        )
    )
    .groupby(
        [
            "dataset",
            "Condition",
            "radius_multiplier"
        ]
    )[summary_columns]
    .median()
    .reset_index()
)

print("\nFlat median summary:")
print(median_summary)

In [ ]:
# ==========================================
# 9. STATISTICAL ANALYSIS BY GRAPH METRIC
#    AND RADIUS
# ==========================================

graph_pairwise_results = []

dataset_definitions = [
    (df1, "Dataset 1", full_order),
    (df2, "Dataset 2", full_order),
    (df_combined, "Combined Dataset", full_order)
]

for column, title, ylabel, filename_prefix in plot_definitions:

    if column not in graph_results.columns:
        print(
            f"Skipping '{column}': "
            "column not found in graph_results."
        )
        continue

    for multiplier in radius_multipliers:

        for df, dataset_name, condition_order in dataset_definitions:

            df_radius = df[
                df["radius_multiplier"] == multiplier
            ].copy()

            if df_radius.empty:
                print(
                    f"\nSkipping {title} at radius "
                    f"{multiplier} — {dataset_name}: no data."
                )
                continue

            # Retrieve the physical radius for this dataset.
            if (
                "radius_um" in df_radius.columns
                and df_radius["radius_um"].notna().any()
            ):
                radius_um = (
                    df_radius["radius_um"]
                    .dropna()
                    .iloc[0]
                )

                radius_description = (
                    f"radius multiplier = {multiplier}, "
                    f"radius = {radius_um:.2f} µm"
                )

            else:
                radius_um = np.nan

                radius_description = (
                    f"radius multiplier = {multiplier}"
                )

            metric_name = (
                f"{title} "
                f"({radius_description})"
            )

            results = run_group_tests(
                df=df_radius,
                value_col=column,
                dataset_name=dataset_name,
                metric_name=metric_name,
                condition_order=condition_order,
                group_col="Condition",
                alpha=0.05
            )

            if results is not None and not results.empty:

                results["radius_multiplier"] = multiplier
                results["radius_um"] = radius_um
                results["value_column"] = column
                results["filename_prefix"] = filename_prefix

                graph_pairwise_results.append(results)


# ==========================================
# 10. COMBINE AND SAVE RESULTS
# ==========================================

if graph_pairwise_results:

    graph_pairwise_results_df = pd.concat(
        graph_pairwise_results,
        ignore_index=True
    )

    preferred_columns = [
        "Metric",
        "value_column",
        "Dataset",
        "radius_multiplier",
        "radius_um",
        "Group column",
        "Comparison",
        "U",
        "Raw_p",
        "Holm_adjusted_p",
        "Significant",
    ]

    graph_pairwise_results_df = (
        graph_pairwise_results_df[
            [
                column
                for column in preferred_columns
                if column in graph_pairwise_results_df.columns
            ]
        ]
    )

    print("\nComplete graph-metric pairwise results:")

    formatters = {}

    if "U" in graph_pairwise_results_df.columns:
        formatters["U"] = "{:.2f}".format

    if "Raw_p" in graph_pairwise_results_df.columns:
        formatters["Raw_p"] = "{:.4g}".format

    if "Holm_adjusted_p" in graph_pairwise_results_df.columns:
        formatters["Holm_adjusted_p"] = "{:.4g}".format

    if "radius_um" in graph_pairwise_results_df.columns:
        formatters["radius_um"] = "{:.3f}".format

    print(
        graph_pairwise_results_df.to_string(
            index=False,
            formatters=formatters
        )
    )

    graph_pairwise_results_df.to_csv(
        "graph_metric_statistical_results_by_radius.csv",
        index=False
    )

    print(
        "\nStatistical results saved to:"
        " graph_metric_statistical_results_by_radius.csv"
    )

else:

    graph_pairwise_results_df = pd.DataFrame()

    print("No pairwise results were generated.")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ==========================================
# 1. SETTINGS
# ==========================================

os.makedirs("Figures", exist_ok=True)

datasets_to_process = [
    1,
    2,
    "combined"
]

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    """Convert condition labels to a consistent format."""
    if pd.isna(label):
        return label

    return label_map.get(
        str(label),
        str(label)
    )


def safe_filename(value):
    """
    Convert a radius multiplier into a filename-safe string.

    For example, 1.5 becomes 1p5.
    """
    value = str(value).replace(
        ".",
        "p"
    )

    value = re.sub(
        r"[^A-Za-z0-9_\-]",
        "_",
        value
    )

    return value


# ==========================================
# 2. GRAPH FEATURES TO INCLUDE
# ==========================================

# Number of lysosomes, cell area, and lysosome
# density are excluded because they are not graph-derived
# metrics and can be analysed separately.
feature_names = [
    "n_edges",
    "mean_degree",
    "graph_density",
    "isolated_fraction",
    "n_components",
    "largest_component_fraction",
    "transitivity"
]

feature_labels = {
    "n_edges": "Number of edges",
    "mean_degree": "Mean degree",
    "graph_density": "Graph density",
    "isolated_fraction": "Isolated fraction",
    "n_components": "Number of components",
    "largest_component_fraction": (
        "Largest component fraction"
    ),
    "transitivity": "Transitivity"
}


# ==========================================
# 3. PREPARE GRAPH RESULTS
# ==========================================

heatmap_results = graph_results.copy()

heatmap_results["Condition"] = (
    heatmap_results["label"]
    .fillna(heatmap_results["tau_state"])
    .apply(standardise_condition)
)

# Ensure all required features are available.
missing_features = [
    feature
    for feature in feature_names
    if feature not in heatmap_results.columns
]

if missing_features:
    raise KeyError(
        "The following features are missing from "
        f"graph_results: {missing_features}"
    )


# Ensure the physical radius is available.
if "radius_um" not in heatmap_results.columns:
    raise KeyError(
        "The 'radius_um' column is missing from "
        "graph_results."
    )


# ==========================================
# 4. CREATE DATASET 1, DATASET 2, AND
#    COMBINED DATA
# ==========================================

df1 = heatmap_results[
    heatmap_results["dataset"].isin([1, "1"])
].copy()

df2 = heatmap_results[
    heatmap_results["dataset"].isin([2, "2"])
].copy()

df_combined = pd.concat(
    [df1, df2],
    ignore_index=True
)

dataset_dataframes = {
    1: df1,
    2: df2,
    "combined": df_combined
}


# ==========================================
# 5. BUILD FEATURE MATRICES
# ==========================================

feature_matrices = {}
standardised_matrices = {}
feature_dataframes = {}

for ds_key in datasets_to_process:

    dataset_df = dataset_dataframes[ds_key].copy()

    # Graph results contain one row per cell and
    # radius multiplier. Matrices are stored
    # separately for every radius.
    feature_matrices[ds_key] = {}
    standardised_matrices[ds_key] = {}
    feature_dataframes[ds_key] = {}

    available_multipliers = sorted(
        dataset_df[
            "radius_multiplier"
        ]
        .dropna()
        .unique()
    )

    for multiplier in available_multipliers:

        radius_df = dataset_df[
            dataset_df["radius_multiplier"]
            == multiplier
        ].copy()

        radius_df = radius_df.sort_values(
            by=[
                "Condition",
                "cell_id"
            ]
        ).reset_index(
            drop=True
        )

        # Keep metadata for identifying rows.
        matrix_df = radius_df[
            [
                "dataset",
                "cell_id",
                "tau_state",
                "label",
                "Condition",
                "radius_multiplier",
                "radius_um"
            ] + feature_names
        ].copy()

        # Remove rows with no values in any graph
        # or spatial feature.
        matrix_df = matrix_df.dropna(
            subset=feature_names,
            how="all"
        ).reset_index(
            drop=True
        )

        # Use a readable row label.
        matrix_df["Row_label"] = (
            matrix_df["Condition"].astype(str)
            + " | cell "
            + matrix_df["cell_id"].astype(str)
        )

        # Raw feature matrix.
        raw_matrix = matrix_df[
            feature_names
        ].astype(float)

        # Feature-wise standardisation.
        # NaN values remain masked in the heatmap.
        feature_mean = raw_matrix.mean(
            axis=0,
            skipna=True
        )

        feature_std = raw_matrix.std(
            axis=0,
            skipna=True
        )

        # Avoid division by zero for constant features.
        feature_std = feature_std.replace(
            0,
            1
        )

        z_matrix = (
            raw_matrix - feature_mean
        ) / feature_std

        # Store all outputs.
        feature_dataframes[ds_key][multiplier] = (
            matrix_df
        )

        feature_matrices[ds_key][multiplier] = (
            raw_matrix
        )

        standardised_matrices[ds_key][multiplier] = (
            z_matrix
        )

        print(
            f"{ds_key}, radius multiplier "
            f"{multiplier}: {raw_matrix.shape}"
        )


# ==========================================
# 6. SAVE FEATURE MATRICES
# ==========================================

for ds_key in datasets_to_process:

    dataset_name = (
        "combined"
        if ds_key == "combined"
        else f"dataset_{ds_key}"
    )

    for multiplier in feature_dataframes[ds_key]:

        matrix_df = (
            feature_dataframes[ds_key][multiplier]
        )

        raw_matrix = (
            feature_matrices[ds_key][multiplier]
        )

        z_matrix = (
            standardised_matrices[ds_key][multiplier]
        )

        radius_name = (
            f"radius_{safe_filename(multiplier)}"
        )

        # Save metadata and raw feature values.
        raw_output = matrix_df[
            [
                "Row_label",
                "Condition",
                "cell_id",
                "radius_multiplier",
                "radius_um"
            ] + feature_names
        ].copy()

        raw_output.to_csv(
            f"Figures/"
            f"{dataset_name}_graph_feature_matrix_"
            f"{radius_name}.csv",
            index=False
        )

        # Save the standardised matrix used for
        # plotting.
        z_output = z_matrix.copy()

        z_output.insert(
            0,
            "Row_label",
            matrix_df[
                "Row_label"
            ].to_numpy()
        )

        z_output.insert(
            1,
            "Condition",
            matrix_df[
                "Condition"
            ].to_numpy()
        )

        z_output.to_csv(
            f"Figures/"
            f"{dataset_name}_standardised_"
            f"graph_feature_matrix_"
            f"{radius_name}.csv",
            index=False
        )


# ==========================================
# 7. HEATMAP FUNCTION
# ==========================================

def create_graph_heatmap(
    ds_key,
    multiplier,
    annotate=False
):
    matrix_df = (
        feature_dataframes[ds_key][multiplier]
    )

    z_matrix = (
        standardised_matrices[ds_key][multiplier]
        .copy()
    )

    dataset_name = (
        "Combined Dataset"
        if ds_key == "combined"
        else f"Dataset {ds_key}"
    )

    radius_name = (
        f"radius_{safe_filename(multiplier)}"
    )

    radius_um = (
        matrix_df["radius_um"]
        .dropna()
        .iloc[0]
        if matrix_df["radius_um"]
        .notna()
        .any()
        else np.nan
    )

    # Rename columns for display only.
    z_matrix = z_matrix.rename(
        columns=feature_labels
    )

    # Use cell and condition as row labels.
    z_matrix.index = matrix_df[
        "Row_label"
    ]

    # The figure height scales with the number
    # of cells.
    figure_height = max(
        4,
        min(
            18,
            0.35 * len(z_matrix) + 2
        )
    )

    fig, ax = plt.subplots(
        figsize=(12, figure_height)
    )

    sns.heatmap(
        z_matrix,
        ax=ax,
        cmap="vlag",
        center=0,
        vmin=-2.5,
        vmax=2.5,
        mask=z_matrix.isna(),
        annot=annotate,
        fmt=".2f",
        linewidths=0.4,
        linecolor="white",
        cbar_kws={
            "label": "Feature-wise z-score"
        }
    )

    ax.set_title(
        f"Graph Feature Matrix — "
        f"{dataset_name}\n"
        f"Radius multiplier = {multiplier}, "
        f"radius = {radius_um:.2f} µm",
        fontsize=14,
        pad=12
    )

    ax.set_xlabel(
        "Graph and spatial features",
        fontsize=12
    )

    ax.set_ylabel(
        "Cell",
        fontsize=12
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.yticks(
        rotation=0
    )

    fig.tight_layout()

    filename = (
        f"Figures/"
        f"{dataset_name.lower().replace(' ', '_')}_"
        f"graph_feature_matrix_{radius_name}.svg"
    )

    fig.savefig(
        filename,
        format="svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {filename}")


# ==========================================
# 8. CREATE ALL HEATMAPS
# ==========================================

for ds_key in datasets_to_process:

    for multiplier in feature_dataframes[ds_key]:

        create_graph_heatmap(
            ds_key=ds_key,
            multiplier=multiplier,
            annotate=False
        )

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA


# ==========================================
# 1. SETTINGS
# ==========================================

os.makedirs("Figures", exist_ok=True)

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

canonical_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

datasets_to_plot = [1, 2, "combined"]


# ==========================================
# 2. STANDARDISE CONDITION LABELS
# ==========================================

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    if pd.isna(label):
        return label

    return label_map.get(
        str(label),
        str(label)
    )


for ds_key in datasets_to_plot:

    if ds_key not in feature_dataframes:
        continue

    for multiplier in feature_dataframes[ds_key]:

        feature_dataframes[ds_key][multiplier][
            "Condition"
        ] = (
            feature_dataframes[ds_key][multiplier]["Condition"]
            .map(standardise_condition)
        )


# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================

def safe_filename(value):
    """Convert a radius multiplier into a filename-safe string."""
    value = str(value).replace(".", "p")

    return "".join(
        character
        if character.isalnum() or character in "_-"
        else "_"
        for character in value
    )


def get_dataset_matrix(
    ds_key,
    multiplier
):
    """
    Return the raw feature matrix and corresponding metadata
    for one dataset and one radius multiplier.
    """

    if ds_key not in feature_dataframes:
        return (
            pd.DataFrame(columns=feature_names),
            pd.DataFrame()
        )

    if multiplier not in feature_dataframes[ds_key]:
        return (
            pd.DataFrame(columns=feature_names),
            pd.DataFrame()
        )

    metadata = feature_dataframes[ds_key][multiplier].copy()

    matrix = metadata[
        feature_names
    ].astype(float).copy()

    return matrix, metadata


def combine_dataset_matrices(multiplier):
    """
    Construct the combined Dataset 1 + Dataset 2 matrix
    and metadata for one radius multiplier.
    """

    matrix_1, metadata_1 = get_dataset_matrix(
        1,
        multiplier
    )

    matrix_2, metadata_2 = get_dataset_matrix(
        2,
        multiplier
    )

    combined_matrix = pd.concat(
        [matrix_1, matrix_2],
        ignore_index=True
    )

    combined_metadata = pd.concat(
        [metadata_1, metadata_2],
        ignore_index=True
    )

    return combined_matrix, combined_metadata


# ==========================================
# 4. PCA AND PLOTTING FOR ONE RADIUS
# ==========================================

def run_pca_for_radius(multiplier):
    """
    Fit PCA once on the combined Dataset 1 + Dataset 2 matrix.

    Dataset 1, Dataset 2, and Combined Dataset are then projected
    using the same imputer, scaler, and PCA transformation.
    """

    # --------------------------------------
    # Build matrices for this radius
    # --------------------------------------

    matrix_1, metadata_1 = get_dataset_matrix(
        1,
        multiplier
    )

    matrix_2, metadata_2 = get_dataset_matrix(
        2,
        multiplier
    )

    matrix_combined, metadata_combined = (
        combine_dataset_matrices(multiplier)
    )

    if matrix_combined.empty:
        print(
            f"Skipping radius {multiplier}: "
            "combined matrix is empty."
        )
        return

    if len(matrix_combined) < 2:
        print(
            f"Skipping radius {multiplier}: "
            "at least two cells are required."
        )
        return

    # --------------------------------------
    # Handle missing graph metrics
    # --------------------------------------
    #
    # Some graph metrics can be undefined, for example:
    # transitivity for graphs containing fewer than three nodes.
    # Median imputation is fitted on the combined matrix only.

    imputer = SimpleImputer(
        strategy="median"
    )

    X_combined_imputed = imputer.fit_transform(
        matrix_combined
    )

    # --------------------------------------
    # Standardise and fit PCA
    # --------------------------------------

    scaler = StandardScaler()

    X_combined_scaled = scaler.fit_transform(
        X_combined_imputed
    )

    pca = PCA(
        n_components=2
    )

    X_combined_pca = pca.fit_transform(
        X_combined_scaled
    )

    # --------------------------------------
    # Transform Dataset 1 and Dataset 2
    # --------------------------------------

    pca_coordinates = {
        "combined": X_combined_pca
    }

    dataset_inputs = {
        1: matrix_1,
        2: matrix_2
    }

    for ds_key, matrix in dataset_inputs.items():

        if matrix.empty:
            pca_coordinates[ds_key] = np.empty(
                (0, 2)
            )
            continue

        X_dataset_imputed = imputer.transform(
            matrix
        )

        X_dataset_scaled = scaler.transform(
            X_dataset_imputed
        )

        pca_coordinates[ds_key] = (
            pca.transform(X_dataset_scaled)
        )

    # --------------------------------------
    # Save PCA loadings
    # --------------------------------------

    pca_loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[
            "PC1_loading",
            "PC2_loading"
        ]
    )

    radius_name = safe_filename(
        multiplier
    )

    pca_loadings.to_csv(
        f"Figures/"
        f"graph_feature_pca_loadings_"
        f"radius_{radius_name}.csv"
    )

    explained_variance = pd.DataFrame({
        "Principal component": [
            "PC1",
            "PC2"
        ],
        "Explained variance ratio": (
            pca.explained_variance_ratio_
        ),
        "Explained variance (%)": (
            pca.explained_variance_ratio_ * 100
        )
    })

    explained_variance.to_csv(
        f"Figures/"
        f"graph_feature_pca_explained_variance_"
        f"radius_{radius_name}.csv",
        index=False
    )

    print(
        f"\n--- PCA: radius multiplier {multiplier} ---"
    )

    print(
        explained_variance.round(3)
    )

    print("\nPCA loadings:")
    print(pca_loadings.round(3))

    # --------------------------------------
    # Save PCA coordinates
    # --------------------------------------

    pca_coordinates_dataframes = {}

    metadata_by_dataset = {
        1: metadata_1,
        2: metadata_2,
        "combined": metadata_combined
    }

    for ds_key in datasets_to_plot:

        coordinates = pca_coordinates[ds_key]
        metadata = metadata_by_dataset[ds_key].copy()

        if coordinates.size == 0:
            pca_coordinates_dataframes[ds_key] = (
                pd.DataFrame(
                    columns=[
                        "PC1",
                        "PC2",
                        "Condition"
                    ]
                )
            )
            continue

        if len(metadata) != len(coordinates):
            raise ValueError(
                f"Metadata and PCA coordinate lengths do not "
                f"match for dataset {ds_key}, "
                f"radius {multiplier}."
            )

        coordinate_df = pd.DataFrame({
            "PC1": coordinates[:, 0],
            "PC2": coordinates[:, 1],
            "Condition": metadata[
                "Condition"
            ].to_numpy(),
            "cell_id": metadata[
                "cell_id"
            ].to_numpy(),
            "radius_multiplier": multiplier
        })

        pca_coordinates_dataframes[ds_key] = (
            coordinate_df
        )

        dataset_name = (
            "combined"
            if ds_key == "combined"
            else f"dataset_{ds_key}"
        )

        coordinate_df.to_csv(
            f"Figures/"
            f"{dataset_name}_graph_feature_pca_coordinates_"
            f"radius_{radius_name}.csv",
            index=False
        )

    # ======================================
    # Plotting function
    # ======================================

    def plot_pca_points(ax, ds_key):
        coordinates = pca_coordinates[ds_key]
        metadata = metadata_by_dataset[ds_key]

        if coordinates.size == 0:
            ax.axis("off")
            return

        current_labels = metadata[
            "Condition"
        ].to_numpy()

        if len(current_labels) != len(coordinates):
            raise ValueError(
                f"Number of labels and PCA coordinates does "
                f"not match for dataset {ds_key}."
            )

        active_categories = (
            order_2
            if ds_key == 2
            else canonical_order
        )

        for condition in active_categories:

            mask = current_labels == condition

            if not np.any(mask):
                continue

            ax.scatter(
                coordinates[mask, 0],
                coordinates[mask, 1],
                label=condition,
                color=traffic_light_map[condition],
                alpha=0.8,
                edgecolors="white",
                linewidth=0.5,
                s=25
            )

        dataset_title = (
            "Combined Dataset"
            if ds_key == "combined"
            else f"Dataset {ds_key}"
        )

        ax.set_title(
            dataset_title,
            fontsize=14,
            pad=10
        )

        ax.set_xlabel(
            f"PC1 "
            f"({pca.explained_variance_ratio_[0]:.1%} variance)",
            fontsize=12
        )

        ax.set_ylabel(
            f"PC2 "
            f"({pca.explained_variance_ratio_[1]:.1%} variance)",
            fontsize=12
        )

        ax.grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        ax.legend(
            frameon=True,
            fontsize=9
        )

    # ======================================
    # Dataset 1 and Dataset 2 plot
    # ======================================

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 3),
        sharex=True,
        sharey=True
    )

    plot_pca_points(
        axes[0],
        1
    )

    plot_pca_points(
        axes[1],
        2
    )

    fig.suptitle(
        f"Graph Feature PCA — "
        f"radius multiplier = {multiplier}",
        fontsize=14
    )

    fig.tight_layout()

    fig.savefig(
        f"Figures/"
        f"graph_features_pca_radius_{radius_name}_"
        f"by_dataset.svg",
        format="svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # ======================================
    # Combined PCA plot
    # ======================================

    fig, ax = plt.subplots(
        figsize=(5, 3)
    )

    plot_pca_points(
        ax,
        "combined"
    )

    ax.set_title(
        f"Combined Graph Feature PCA\n"
        f"radius multiplier = {multiplier}",
        fontsize=14,
        pad=10
    )

    fig.tight_layout()

    fig.savefig(
        f"Figures/"
        f"graph_features_pca_radius_{radius_name}_"
        f"combined.svg",
        format="svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return {
        "pca": pca,
        "scaler": scaler,
        "imputer": imputer,
        "coordinates": pca_coordinates_dataframes,
        "loadings": pca_loadings,
        "explained_variance": explained_variance
    }


# ==========================================
# 5. RUN PCA FOR ALL RADIUS MULTIPLIERS
# ==========================================

pca_results = {}

all_radius_multipliers = sorted(
    set(df1["radius_multiplier"].dropna().unique())
    | set(df2["radius_multiplier"].dropna().unique())
)

for multiplier in all_radius_multipliers:

    pca_results[multiplier] = run_pca_for_radius(
        multiplier
    )